# 🤖 Machine Learning Classification Assignment
### Auto Dataset Download → Preprocess → Model → Graphs → GitHub

---
> **📌 Aapko kuch bhi manually download nahi karna. Bas Step 1 mein apni Kaggle API key daalo — baaki sab khud hoga!**
---

## 📦 Step 0: Install Libraries

In [ ]:
!pip install kaggle pandas numpy scikit-learn matplotlib seaborn --quiet
print('✅ Libraries ready!')

---
## 📥 Step 1: Auto-Download Dataset from Kaggle API

### 🔑 Kaggle API Key kaise milegi?
1. [kaggle.com](https://www.kaggle.com) par login karo
2. Top-right → **Profile picture** → **Settings**
3. **API** section mein → **Create New Token** click karo
4. `kaggle.json` download hoga — us mein se `username` aur `key` yahan paste karo 👇

In [ ]:
import os, json, zipfile, glob

# ╔══════════════════════════════════════════════╗
# ║   🔑  SIRF YEH DO CHEEZEIN CHANGE KARO      ║
# ╚══════════════════════════════════════════════╝
KAGGLE_USERNAME = "your_kaggle_username"   # ← apna username
KAGGLE_KEY      = "your_kaggle_api_key"    # ← apni key

# Dataset choices (ek uncomment karo):
DATASET = "cicdataset/cicids2017"          # ~2.2 GB — Network Intrusion Detection
# DATASET = "mlg-ulb/creditcardfraud"      # ~150 MB — Credit Card Fraud
# DATASET = "uciml/forest-cover-type-dataset" # ~75 MB — Forest Cover
# DATASET = "fedesoriano/stroke-prediction-dataset" # ~1 MB — Stroke

# ── Setup credentials automatically ──────────────────────────
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
cred_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(cred_path, 'w') as f:
    json.dump(creds, f)
os.chmod(cred_path, 0o600)
print(f'✅ Credentials set for: {KAGGLE_USERNAME}')

# ── Download dataset automatically ───────────────────────────
DATA_DIR = 'dataset'
os.makedirs(DATA_DIR, exist_ok=True)

print(f'⬇️  Downloading: {DATASET} ...')
!kaggle datasets download -d {DATASET} -p {DATA_DIR}

# ── Unzip all downloaded files ───────────────────────────────
for zf in glob.glob(f'{DATA_DIR}/*.zip'):
    print(f'📂 Unzipping: {zf}')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(zf)

print('\n✅ Download complete! Files:')
total_size = 0
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        fp = os.path.join(root, file)
        sz = os.path.getsize(fp)
        total_size += sz
        print(f'  📄 {fp}  ({sz/1024**2:.1f} MB)')
print(f'\n📦 Total size: {total_size/1024**3:.3f} GB')

---
## 📖 Step 2: Read the Dataset

In [ ]:
import pandas as pd
import numpy as np

# Auto-detect and load all CSV files
csv_files = glob.glob('dataset/**/*.csv', recursive=True)
print(f'Found {len(csv_files)} CSV file(s)')

dfs = []
for f in csv_files:
    print(f'Reading: {f}')
    try:
        temp = pd.read_csv(f, low_memory=False)
        dfs.append(temp)
        print(f'  → Shape: {temp.shape}')
    except Exception as e:
        print(f'  ⚠️ Skip: {e}')

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f'\n✅ Full Dataset Shape: {df.shape}')
print(f'Memory Usage: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
df.head(3)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Basic Stats ===')
df.describe()

---
## 🧹 Step 3: Preprocessing

In [ ]:
# ── 3.1 Strip whitespace from column names ────────────────────
df.columns = df.columns.str.strip()

# ── 3.2 Replace inf values ────────────────────────────────────
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# ── 3.3 Missing values ────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Missing values per column:')
print(missing if not missing.empty else '✅ No missing values!')

# Drop columns >50% missing
thresh = len(df) * 0.5
df.dropna(thresh=thresh, axis=1, inplace=True)

# Fill remaining
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

# ── 3.4 Remove duplicates ─────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df):,} duplicate rows')
print(f'Final shape: {df.shape}')

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── 3.5 Set your target column ────────────────────────────────
# Common names: 'Label', 'label', 'Class', 'target', 'Attack'
TARGET = ' Label'   # CICIDS uses ' Label' (with space)
# ⚠️  Agar error aaye to: TARGET = df.columns[-1]

if TARGET not in df.columns:
    TARGET = df.columns[-1]
    print(f'⚠️  Using last column as target: "{TARGET}"')

print(f'Target: "{TARGET}"')
print(df[TARGET].value_counts())

# ── 3.6 Encode target ─────────────────────────────────────────
le = LabelEncoder()
if df[TARGET].dtype == 'object':
    class_names = df[TARGET].unique().tolist()
    df[TARGET] = le.fit_transform(df[TARGET])
    print('\nEncoded classes:', list(le.classes_))

# ── 3.7 Encode other categoricals ────────────────────────────
for col in cat_cols:
    if col in df.columns and col != TARGET:
        df[col] = le.fit_transform(df[col].astype(str))

# ── 3.8 Feature/Target split + scale ─────────────────────────
X = df.drop(columns=[TARGET])
y = df[TARGET]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nX shape: {X_scaled.shape}')
print(f'y shape: {y.shape}')
print('✅ Preprocessing done!')

---
## 🤖 Step 4: Classification Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

# Models
models = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':        DecisionTreeClassifier(random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors':  KNeighborsClassifier(n_neighbors=5),
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'accuracy': acc, 'time': elapsed, 'y_pred': y_pred}
    trained_models[name] = model
    print(f'  ✅ Accuracy: {acc:.4f} | Time: {elapsed:.2f}s')

# Best model
best_name = max(results, key=lambda k: results[k]['accuracy'])
print(f'\n🏆 Best Model: {best_name} ({results[best_name]["accuracy"]:.4f})')
print(classification_report(y_test, results[best_name]['y_pred']))

---
## 📊 Step 5: Results in Graphs

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

names = list(results.keys())
accuracies = [results[n]['accuracy'] for n in names]
times = [results[n]['time'] for n in names]

# ── Graph 1: Class Distribution ───────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
vc = y.value_counts()
ax[0].bar(vc.index.astype(str), vc.values, color=sns.color_palette('Set2', len(vc)))
ax[0].set_title('Class Distribution', fontweight='bold')
ax[0].set_xlabel('Class'); ax[0].set_ylabel('Count')
ax[1].pie(vc.values, labels=vc.index.astype(str), autopct='%1.1f%%',
          colors=sns.color_palette('Set2', len(vc)))
ax[1].set_title('Class Distribution (Pie)', fontweight='bold')
plt.tight_layout(); plt.savefig('graph1_class_dist.png', bbox_inches='tight'); plt.show()
print('✅ Graph 1 saved')

In [ ]:
# ── Graph 2: Model Accuracy Comparison ───────────────────────
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bars = ax[0].barh(names, accuracies, color=sns.color_palette('viridis', len(names)))
ax[0].set_xlim(0, 1.1); ax[0].set_title('Model Accuracy', fontweight='bold')
for bar, acc in zip(bars, accuracies):
    ax[0].text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
               f'{acc:.4f}', va='center', fontweight='bold')
ax[1].barh(names, times, color=sns.color_palette('rocket_r', len(names)))
ax[1].set_title('Training Time (sec)', fontweight='bold')
plt.tight_layout(); plt.savefig('graph2_accuracy.png', bbox_inches='tight'); plt.show()
print('✅ Graph 2 saved')

In [ ]:
# ── Graph 3: Confusion Matrix (Best Model) ───────────────────
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig('graph3_confusion.png', bbox_inches='tight'); plt.show()
print('✅ Graph 3 saved')

In [ ]:
# ── Graph 4: Feature Importance ──────────────────────────────
if 'Random Forest' in trained_models:
    rf = trained_models['Random Forest']
    feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_})
    feat_imp = feat_imp.sort_values('Importance', ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feat_imp['Feature'][::-1], feat_imp['Importance'][::-1],
            color=sns.color_palette('magma_r', 15))
    ax.set_title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout(); plt.savefig('graph4_feat_importance.png', bbox_inches='tight'); plt.show()
    print('✅ Graph 4 saved')

In [ ]:
# ── Graph 5: Correlation Heatmap ─────────────────────────────
top_feats = feat_imp['Feature'].tolist() if 'feat_imp' in dir() else X.columns[:12].tolist()
fig, ax = plt.subplots(figsize=(12, 8))
corr = df[top_feats].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout(); plt.savefig('graph5_correlation.png', bbox_inches='tight'); plt.show()
print('✅ Graph 5 saved')

In [ ]:
# ── Graph 6: ROC Curves ───────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import label_binarize
n_classes = len(np.unique(y))

fig, ax = plt.subplots(figsize=(8, 6))
for name, model in trained_models.items():
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)
        if n_classes == 2:
            fpr, tpr, _ = roc_curve(y_test, y_prob[:,1])
            auc = roc_auc_score(y_test, y_prob[:,1])
            ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')
        else:
            y_bin = label_binarize(y_test, classes=np.unique(y))
            auc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
            ax.plot([0,1],[0,1], alpha=0)
            ax.text(0.5, 0.05+list(trained_models).index(name)*0.06,
                    f'{name}: AUC={auc:.3f}', fontsize=9)
ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold'); ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig('graph6_roc.png', bbox_inches='tight'); plt.show()
print('✅ Graph 6 saved')

In [ ]:
# ── Graph 7: Final Dashboard ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 ML Classification Results Dashboard', fontsize=18, fontweight='bold')

axes[0,0].bar(names, accuracies, color=sns.color_palette('Set2', len(names)))
axes[0,0].set_title('Accuracy by Model'); axes[0,0].set_ylim(0,1)
axes[0,0].tick_params(axis='x', rotation=15); axes[0,0].set_ylabel('Accuracy')

sns.heatmap(confusion_matrix(y_test, results[best_name]['y_pred']),
            annot=True, fmt='d', cmap='YlOrRd', ax=axes[0,1])
axes[0,1].set_title(f'Best: {best_name}'); axes[0,1].set_xlabel('Pred'); axes[0,1].set_ylabel('Actual')

axes[1,0].bar(vc.index.astype(str), vc.values, color=sns.color_palette('pastel'))
axes[1,0].set_title('Class Distribution'); axes[1,0].set_xlabel('Class')

axes[1,1].bar(names, times, color=sns.color_palette('flare', len(names)))
axes[1,1].set_title('Training Time (s)'); axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('graph7_dashboard.png', bbox_inches='tight', dpi=150); plt.show()
print('✅ Graph 7 (Dashboard) saved')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────
summary = pd.DataFrame({
    'Model': names,
    'Accuracy': [f"{results[n]['accuracy']:.4f}" for n in names],
    'Time (s)': [f"{results[n]['time']:.2f}" for n in names]
}).sort_values('Accuracy', ascending=False)

print('='*50)
print('       FINAL RESULTS SUMMARY')
print('='*50)
print(summary.to_string(index=False))
print('='*50)
print(f'🏆 Best: {best_name}  |  Accuracy: {results[best_name]["accuracy"]:.4f}')

---
## 🚀 Step 6: Push to GitHub

In [ ]:
# ── Create README.md ──────────────────────────────────────────
readme = f"""# 🤖 ML Classification Assignment

End-to-end ML pipeline — auto dataset download, preprocessing, 5 models, 7 graphs.

## Dataset
- **Source:** Kaggle — `{DATASET}`
- **Size:** {total_size/1024**3:.2f} GB

## Models
| Model | Accuracy |
|-------|----------|
""" + '\n'.join([f"| {n} | {results[n]['accuracy']:.4f} |" for n in names]) + f"""

## Best Model: {best_name} — {results[best_name]['accuracy']:.4f}

## How to Run
```bash
git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
cd YOUR_REPO
pip install -r requirements.txt
jupyter notebook ML_Classification_Assignment.ipynb
```
Set your Kaggle credentials in Step 1 — dataset downloads automatically!

## Graphs
1. Class Distribution | 2. Model Accuracy | 3. Confusion Matrix
4. Feature Importance | 5. Correlation Heatmap | 6. ROC Curves | 7. Dashboard
"""
with open('README.md','w') as f: f.write(readme)

with open('requirements.txt','w') as f:
    f.write('pandas\nnumpy\nscikit-learn\nmatplotlib\nseaborn\nkaggle\njupyter\n')

with open('.gitignore','w') as f:
    f.write('dataset/\n*.csv\n*.zip\n__pycache__/\n.kaggle/\n.ipynb_checkpoints/\n')

print('✅ README.md, requirements.txt, .gitignore created!')
print('\n📌 Now run these in terminal:')
print('git init')
print('git add .')
print('git commit -m "ML Classification Assignment"')
print('git remote add origin https://github.com/YOUR_USERNAME/YOUR_REPO.git')
print('git push -u origin main')

---
## 📤 Step 7: Group mein share karo

```
📌 ML Classification Assignment
👤 Name: [Apna Naam]
📂 GitHub: https://github.com/YOUR_USERNAME/YOUR_REPO
📊 Dataset: CICIDS 2017 (~2.2 GB) — Kaggle
🤖 Best Model: [Model Name] — Accuracy: XX%
```

---
## ✅ Checklist

| # | Task | Status |
|---|------|--------|
| 1 | Dataset auto-downloaded (GB scale) | ☐ |
| 2 | Dataset read | ☐ |
| 3 | Preprocessed | ☐ |
| 4 | 5 models trained | ☐ |
| 5 | 7 graphs produced | ☐ |
| 6 | Pushed to GitHub | ☐ |
| 7 | Link shared in group | ☐ |